# Лабораторная работа 5

## Салятов Сергей, Подосенов Андрей, M3337

## Вариант 4

В нашем датасете приведены данные о музыкальных произведениях.

Построим линейную модель, где в качестве независимых переменных выступают продолжительность (_song_duration_ms_), "танцевальность" (_danceability_), энергичность (_energy_) и константа, а зависимой - популярность (_song_popularity_)

In [47]:
import numpy as np
import pandas as pd

data_frame = pd.read_csv("song_data.csv")

data_frame['song_duration_sec'] = data_frame['song_duration_ms'] / 1000.0

n = data_frame.shape[0]
k = 4
X = np.zeros((n, 4))

X[:, 0] = 1  # константа (все единицы)
X[:, 1] = data_frame['song_duration_sec'].values  # продолжительность в секундах
X[:, 2] = data_frame['danceability'].values  # танцевальность
X[:, 3] = data_frame['energy'].values  # энергичность

y = data_frame['song_popularity'].values

### Оценка коэффициентов и остаточной дисперсии

По оценке метода наименьших квадратов имеем точную формулу для оценки $\hat{b}$:

$$
\hat{b} = (X^TX)^{-1}X^TY
$$

Также через неё выразим оценку для остаточной несмещённой дисперсии:

$$
S^2(\hat{b}) = (Y - X\hat{b})^T(Y - X\hat{b})
$$

$$
\hat{\sigma} = \frac{S^2(\hat{b})}{n - k}
$$

In [48]:
XTX = X.T @ X
XTX_inv = np.linalg.inv(XTX)
beta_hat = XTX_inv @ (X.T @ y)

print("\nОценки коэффициентов β̂:")
print(f"β₀ (константа)     = {beta_hat[0]:.4f}")
print(f"β₁ (duration)       = {beta_hat[1]:.6f}")
print(f"β₂ (danceability)   = {beta_hat[2]:.4f}")
print(f"β₃ (energy)         = {beta_hat[3]:.4f}")


y_hat = X @ beta_hat                # предсказанные значения ŷ
residuals = y - y_hat               # остатки e = y - ŷ
rss = np.sum(residuals**2)          # RSS = Σ e_i²
sigma2_hat = rss / (n - k)          # несмещённая оценка дисперсии

print(f"\nRSS (сумма квадратов остатков) = {rss:.4f}")
print(f"Остаточная дисперсия σ̂²         = {sigma2_hat:.4f}")


Оценки коэффициентов β̂:
β₀ (константа)     = 44.6097
β₁ (duration)       = -0.002850
β₂ (danceability)   = 14.4782
β₃ (energy)         = -0.2567

RSS (сумма квадратов остатков) = 8938708.5935
Остаточная дисперсия σ̂²         = 474.6805


### Доверительные интервалы 

Из статистик 

$$
\sqrt{n - k} \cdot \frac{\hat{b_j} - b_j}{S(\hat{b_j}) \sqrt{A_{jj}^{-1}}} \sim T(n-k)
$$

и

$$
\frac{S^2(\hat{b})}{\sigma^2} \sim \chi^2(n-k)
$$

Можем вывести доверительные интервалы для коэффициентов и дисперсии уровня $\alpha$:

$$
b_j \in \hat{b}_j \pm t_{1-\alpha/2,\,n-k} \cdot \hat{\sigma} \sqrt{A_{jj}^{-1}}
$$

$$
\sigma^2 \in \left[ \frac{(n - k) \hat{\sigma}^2}{\chi^2_{1-\alpha/2,\,n-k}},\; \frac{(n - k) \hat{\sigma}^2}{\chi^2_{\alpha/2,\,n-k}} \right]
$$

In [49]:
from scipy.stats import t, chi2

# Доверительные интервалы для коэффициентов
alpha = 0.05
df = n - k  # степени свободы

# Стандартные ошибки коэффициентов
se_beta = np.sqrt(sigma2_hat * np.diag(XTX_inv))

# Критическое значение t для 95% ДИ
t_crit = t.ppf(1 - alpha/2, df)

# Границы доверительных интервалов
ci_lower = beta_hat - t_crit * se_beta
ci_upper = beta_hat + t_crit * se_beta

print("\n95%-е доверительные интервалы для коэффициентов:")
for j, name in enumerate(["Константа", "Duration", "Danceability", "Energy"]):
    print(f"{name}: [{ci_lower[j]:.4f}, {ci_upper[j]:.4f}]")

# Доверительный интервал для дисперсии
chi2_lower = chi2.ppf(alpha/2, df)      # левый квантиль
chi2_upper = chi2.ppf(1 - alpha/2, df)  # правый квантиль

ci_sigma2_lower = (df * sigma2_hat) / chi2_upper
ci_sigma2_upper = (df * sigma2_hat) / chi2_lower

print(f"\n95%-й доверительный интервал для дисперсии ошибок σ²:")
print(f"[{ci_sigma2_lower:.4f}, {ci_sigma2_upper:.4f}]")

# Для стандартного отклонения (опционально)
ci_sigma_lower = np.sqrt(ci_sigma2_lower)
ci_sigma_upper = np.sqrt(ci_sigma2_upper)
print(f"\n95%-й доверительный интервал для стандартного отклонения σ:")
print(f"[{ci_sigma_lower:.4f}, {ci_sigma_upper:.4f}]")


95%-е доверительные интервалы для коэффициентов:
Константа: [42.6403, 46.5790]
Duration: [-0.0081, 0.0024]
Danceability: [12.4787, 16.4777]
Energy: [-1.7186, 1.2052]

95%-й доверительный интервал для дисперсии ошибок σ²:
[465.2366, 484.4163]

95%-й доверительный интервал для стандартного отклонения σ:
[21.5693, 22.0095]


### Коэффициент детерминации $R^2$

Коэффициент детерминации $R^2$ определяется как квадрат коэффициента корреляции между истинными значениями $y$ и предсказанными значениями $\hat{y}$:

$$
R^2 = \left( \frac{\displaystyle{\sum_{i=1}^n} (y_i - \bar{y})(\hat{y}_i - \bar{y})}{\sqrt{\displaystyle{\sum_{i=1}^n} (y_i - \bar{y})^2} \sqrt{\displaystyle{\sum_{i=1}^n} (\hat{y}_i - \bar{y})^2}} \right)^2
$$

где:
- $y_i$ — истинные значения,
- $\hat{y}_i$ — предсказанные значения,
- $\bar{y}$ — среднее значение $y$.

In [50]:
# 1. Вычисляем среднее значение y
y_mean = np.mean(y)

# 2. Вычисляем компоненты формулы
numerator = np.sum((y - y_mean) * (y_hat - y_mean))
denominator = np.sqrt(np.sum((y - y_mean)**2)) * np.sqrt(np.sum((y_hat - y_mean)**2))

# 3. Коэффициент корреляции R
R = numerator / denominator

# 4. Коэффициент детерминации R²
R2 = R ** 2

print(f"\nКоэффициент корреляции R = {R:.4f}")
print(f"Коэффициент детерминации R² = {R2:.4f}")

# Для проверки: стандартный метод R² = 1 - RSS/TSS
TSS = np.sum((y - y_mean)**2)
R2_standard = 1 - (rss / TSS)
print(f"\nПроверка через стандартную формулу R² = 1 - RSS/TSS: {R2_standard:.4f}")


Коэффициент корреляции R = 0.1046
Коэффициент детерминации R² = 0.0109

Проверка через стандартную формулу R² = 1 - RSS/TSS: 0.0109


Очень низкий $R^2$ указывает, что выбранные переменные слабо объясняют популярность, возможно, из-за пропуска важных предикторов или несущественности самих признаков

### Проверка гипотез

- Чем больше энергичность, тем больше популярность

    Нет, коэффициент перед признаком _energy_ равен -0.2567, то есть при увеличении показателя энергии на 1 пункт, популярность уменьшается на 0.2567 единиц

- Популярность зависит от продолжительности

    Проведём t-тест для коэффициента _song_duration_sec_

    $H_0$: $\beta_{duaration} = 0$

    $H_1$: $\beta_{duaration} \neq 0$

In [51]:
t_duration_stat = beta_hat[1] / se_beta[1]

p_value_duration = 2 * (1 - t.cdf(abs(t_duration_stat), df=df))

# Критическое значение для двустороннего теста при alpha=0.05
t_crit = t.ppf(1 - alpha/2, df=df)

# Вывод результатов
print(f"\nT-тест для коэффициента при duration:")
print(f"  Коэффициент (β_duration) = {beta_hat[1]:.6f}")
print(f"  Стандартная ошибка = {se_beta[1]:.6f}")
print(f"  t-статистика = {t_duration_stat:.4f}")
print(f"  p-value (двусторонний) = {p_value_duration:.4f}")
print(f"  Критическое значение (α={alpha}) = ±{t_crit:.3f}")

# Принятие решения
if p_value_duration < alpha:
    print("  → Отвергаем H0: коэффициент статистически значим (продолжительность влияет на популярность)")
else:
    print("  → Не отвергаем H0: коэффициент статистически незначим (нет доказательств влияния продолжительности)")


T-тест для коэффициента при duration:
  Коэффициент (β_duration) = -0.002850
  Стандартная ошибка = 0.002679
  t-статистика = -1.0641
  p-value (двусторонний) = 0.2873
  Критическое значение (α=0.05) = ±1.960
  → Не отвергаем H0: коэффициент статистически незначим (нет доказательств влияния продолжительности)


То есть по результатам t-теста популярность НЕ зависит от продолжительности, так как коэффициент незначим и не отвергаем $H_0$

- Проверить гипотезу $H_0$ о равенстве одновременно нулю коэффициентов при энергичности и "танцевальности".

    $H_1$ - отрицание $H_0$

    Для этого проведём F-тест, где в ограниченной модели будет только константа и _song_duration_sec_. 
    
    Статистика F-теста:

    $$
    F = \frac{n - k}{q} \cdot \frac{S^2(\hat{b}_{T,t_0}) - S^2(\hat{b})}{S^2(\hat{b})} \sim F(q, n-k)
    $$

    где 
    - $F(q, n-k)$ - распределение Фишера со степенями свободы $q$ и $n-k$ 
    - $q$ = 2 - число ограничений
    - $n-k$ - число степеней свободы изначальной модели
    - $S^2(\hat{b})$ - остаточная сумма квадратов полной модели
    - $S^2(\hat{b}_{T,t_0})$ - остаточная сумма квадратов модели с ограничениями (матрица $T$ линейного преобразования оставляет только константу и продолжительность, а вектор $t_0$ равен 0 для проверки гипотезы)

    $$
    T = \begin{bmatrix}
    0 & 0 & 1 & 0 \\
    0 & 0 & 0 & 1
    \end{bmatrix}, \quad
    t_0 = \begin{bmatrix}
    0 \\
    0
    \end{bmatrix}
    $$

In [ ]:
from scipy.stats import f

# 1. Параметры
q = 2  # количество ограничений (energy + danceability)
df_full = df  # степени свободы полной модели (n - 4)

# 2. Построение ограниченной модели (без energy и danceability)
X_restricted = np.column_stack([
    np.ones(n), 
    data_frame['song_duration_sec'].values
])

# 3. Оценка коэффициентов для ограниченной модели
XTX_r = X_restricted.T @ X_restricted
beta_hat_r = np.linalg.inv(XTX_r) @ (X_restricted.T @ y)

# 4. Расчёт RSS для обеих моделей
y_hat_full = X @ beta_hat
residuals_full = y - y_hat_full
rss_full = np.sum(residuals_full**2)

y_hat_restricted = X_restricted @ beta_hat_r
residuals_restricted = y - y_hat_restricted
rss_restricted = np.sum(residuals_restricted**2)

# 5. F-статистика
F_stat = ((rss_restricted - rss_full) / q) / (rss_full / df_full)

# 6. p-value
p_value_F = 1 - f.cdf(F_stat, dfn=q, dfd=df_full)

# 7. Вывод результатов
print("\nF-тест для совместной гипотезы:")
print(f"  RSS полной модели = {rss_full:.4f}")
print(f"  RSS ограниченной модели = {rss_restricted:.4f}")
print(f"  F-статистика = {F_stat:.4f}")
print(f"  p-value = {p_value_F:.6f}")
print(f"  Критическое значение F_{q},{df_full} (α=0.05) = {f.ppf(0.95, q, df_full):.4f}")

# 8. Принятие решения
if p_value_F < 0.05:
    print("  → Отвергаем H0: коэффициенты совместно значимы")
else:
    print("  → Не отвергаем H0: коэффициенты совместно незначимы")


F-тест для совместной гипотезы:
  RSS полной модели = 8938708.5935
  RSS ограниченной модели = 9034411.6245
  F-статистика = 100.8078
  p-value = 0.000000
  Критическое значение F_2,18831 (α=0.05) = 2.9962
  → Отвергаем H0: коэффициенты совместно значимы


То есть одновременно коэффициенты перед признаками энергичность и "танцевальность" не могут быть равны 0.

(Длаее встроенная линейная модель для сверки некоторых результатов)

In [53]:
import statsmodels.api as sm

df = pd.read_csv("song_data.csv")
df['song_duration_sec'] = df['song_duration_ms'] / 1000.0

# Подготовка данных
X = df[["song_duration_sec", "danceability", "energy"]]
y = df["song_popularity"]
X = sm.add_constant(X)

# Построение модели
model = sm.OLS(y, X).fit()

# Вывод результатов
print("=== Модель ===")
print(model.summary())
print("\nДоверительные интервалы:")
print(model.conf_int())
print("\nR²:", model.rsquared)

# Проверка гипотез
print("\n=== Проверка гипотез ===")
# Гипотеза 1 (односторонний тест для energy)
p_energy = model.pvalues["energy"]
p_one_sided = p_energy / 2 if model.params["energy"] > 0 else 1 - p_energy / 2
print("Гипотеза 1 (energy > 0): p-value =", p_one_sided)

# Гипотеза 2
p_duration = model.pvalues["song_duration_sec"]
print("Гипотеза 2 (duration ≠ 0): p-value =", p_duration)

# Гипотеза 3
hypotheses = ["energy = 0", "danceability = 0"]
f_test = model.f_test(hypotheses)
print("Гипотеза 3 (совместная проверка): p-value =", f_test.pvalue)

=== Модель ===
                            OLS Regression Results                            
Dep. Variable:        song_popularity   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     69.47
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           1.13e-44
Time:                        14:00:10   Log-Likelihood:                -84760.
No. Observations:               18835   AIC:                         1.695e+05
Df Residuals:                   18831   BIC:                         1.696e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                44